In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("/Users/premthatikonda/Downloads/FeatureEnggAndMLOPs/data/raw/Unit-02/instagram_posts.csv")
df.head()

,post_id,creator_id,creator_follower_count,creator_follower_count_copy,creator_avg_past_likes,is_reel,video_duration_sec,posting_hour,posting_day,hashtag_count,...,caption_length,engagement_rate,likes_count,comments_count,shares_count,saves_count,views_count,went_viral,random_noise_1,random_noise_2_device
0,20018,206,1587,1657.541767,85,0,0.0,20,Thu,10,...,40,0.23681,282,50,21,35,2072,1,37.445583,iOS
1,20099,40,21512,21495.868570,564,0,0.0,19,Sun,8,...,44,0.05653,870,118,132,117,13283,0,35.429821,Android
2,20790,89,218677,218745.815777,8649,1,21.7,22,Mon,9,...,41,0.00300,525,88,60,62,656902,0,43.750264,Android
3,20631,8,58990,58980.196307,1837,1,31.7,21,Fri,8,...,40,0.03152,1284,196,162,166,237851,0,49.619502,iOS
4,20811,102,7438,7430.605230,212,0,0.0,17,Fri,8,...,86,0.10675,591,91,75,52,9979,0,67.078109,iOS


In [3]:
work = df.copy()
work["log_followers"] = np.log1p(work["creator_follower_count"])
work["log_followers_copy"] = np.log1p(work["creator_follower_count_copy"])
work = pd.get_dummies(work, columns=["caption_category"], prefix="cat" ,drop_first=True)
work

,post_id,creator_id,creator_follower_count,creator_follower_count_copy,creator_avg_past_likes,is_reel,video_duration_sec,posting_hour,posting_day,hashtag_count,...,views_count,went_viral,random_noise_1,random_noise_2_device,log_followers,log_followers_copy,cat_Motivational,cat_Personal,cat_Promotional,cat_Question
0,20018,206,1587,1657.541767,85,0,0.0,20,Thu,10,...,2072,1,37.445583,iOS,7.370231,7.413694,True,False,False,False
1,20099,40,21512,21495.868570,564,0,0.0,19,Sun,8,...,13283,0,35.429821,Android,9.976413,9.975663,False,False,False,True
2,20790,89,218677,218745.815777,8649,1,21.7,22,Mon,9,...,656902,0,43.750264,Android,12.295356,12.295670,False,True,False,False
3,20631,8,58990,58980.196307,1837,1,31.7,21,Fri,8,...,237851,0,49.619502,iOS,10.985140,10.984974,True,False,False,False
4,20811,102,7438,7430.605230,212,0,0.0,17,Fri,8,...,9979,0,67.078109,iOS,8.914492,8.913497,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1395,20212,9,35979,35990.086101,1099,1,26.5,9,Thu,4,...,140970,0,56.001806,Android,10.490719,10.491027,False,False,True,False
1396,20503,29,107439,107378.762119,3479,0,0.0,16,Wed,10,...,149630,0,60.253080,Web,11.584688,11.584127,False,True,False,False
1397,20538,102,7438,7444.726774,212,0,0.0,23,Mon,9,...,9884,0,56.631526,iOS,8.914492,8.915396,True,False,False,False
1398,21221,170,52117,52137.895693,922,0,0.0,10,Sat,10,...,54298,0,50.620871,Web,10.861266,10.861667,False,False,False,False


In [6]:
feature_cols = (
    ['log_followers', 'log_followers_copy', "is_reel", "video_duration_sec", "posting_hour", "hashtag_count", 
     "num_mentions", "caption_length", "random_noise_1" ] 
     + [c for c in work.columns if c.startswith("cat_")]
)

X = work[feature_cols]
Y = work["engagement_rate"]

print(f"{len(feature_cols)} candidate features: {feature_cols}")

13 candidate features: ['log_followers', 'log_followers_copy', 'is_reel', 'video_duration_sec', 'posting_hour', 'hashtag_count', 'num_mentions', 'caption_length', 'random_noise_1', 'cat_Motivational', 'cat_Personal', 'cat_Promotional', 'cat_Question']


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training shape:", X_train_scaled.shape)
print("Testing shape:", X_test_scaled.shape)

Training shape: (1120, 13)
Testing shape: (280, 13)


## ElasticNet CV

In [9]:
from sklearn.linear_model import ElasticNetCV

elastic = ElasticNetCV(random_state=42, max_iter=1000, l1_ratio=[.1, .3, .5, .7, .9], cv=5)
elastic.fit(X_train_scaled, Y_train)

elastic_coefs = pd.DataFrame({"feature": feature_cols, "coefficient": elastic.coef_}).sort_values(by="coefficient", key=abs, ascending=False)
elastic_coefs

,feature,coefficient
0,log_followers,-0.037110
1,log_followers_copy,-0.033959
2,is_reel,0.007884
4,posting_hour,0.007207
11,cat_Promotional,-0.002583
9,cat_Motivational,-0.000955
12,cat_Question,0.000885
5,hashtag_count,0.000802
7,caption_length,-0.000739
10,cat_Personal,-0.000463
